# 基於時空特化特徵之 R-Tree 空間森林建構與分區視覺化
本筆記本展示全新設計的空間森林建構算法：
1. **異常資料移除**：使用滑動窗口與日歸零檢測排除收集端故障網格。
2. **空間自組織根部劃分 (Spatiotemporal Root Partitioning)**：結合地理鄰近限制（KNN 連通圖）與原始人流日特徵曲線（Pearson 相關性），使用 Ward 凝聚階層聚類將城市切分為 $K$ 個彼此特徵差異最大且地理連續的「根區域 (Root Regions)」。
3. **區域內 MBR 迭代建樹**：在各個根區域內部，自底向上迭代聚合成 MBR 樹，建立微觀（Level 1）到宏觀（Level L）的空間層級結構。
4. **視覺化證明**：繪製分區地圖、日特徵曲線，並以 `matplotlib` (`plt`) 證明分區的合理性與時間相位差異。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import AgglomerativeClustering
import os
import sys
import gc

# ── Global plot style: Times New Roman, publication-quality ──
matplotlib.rcParams.update({
    "font.family":        "Times New Roman",
    "font.size":          14,
    "axes.titlesize":     15,
    "axes.labelsize":     14,
    "xtick.labelsize":    12,
    "ytick.labelsize":    12,
    "legend.fontsize":    12,
    "figure.dpi":         120,
    "axes.unicode_minus": False,
})

# ── Global model / data hyperparameters (declared here once) ──
NUM_GRIDS = 500   # number of grids to select from clean data
LOOKBACK  = 12    # input sequence length (hours)
K_ROOTS   = 4     # number of root spatial partitions

print(f"Config: NUM_GRIDS={NUM_GRIDS}, LOOKBACK={LOOKBACK}, K_ROOTS={K_ROOTS}")
print("Environment loaded successfully.")


## 1. 數據載入與異常網格排除

In [ ]:
df_raw = pd.read_parquet('data/2019_hourly_pop_byCHT.parquet')

In [ ]:


grids_raw = df_raw['Grid'].values
data_raw = df_raw.iloc[:, 1:].values.astype(np.float32)
N_raw, T_raw = data_raw.shape
print(f"載入原始資料: {N_raw} 個網格, {T_raw} 個小時。")

# 分離出經緯度座標
coords = np.array([[float(g.split('_')[0]), float(g.split('_')[1])] for g in grids_raw])

# 2. 檢測類型 1: 最後 24 小時為 0 (k_hours = 24)
k_hours = 24
type1_indices = []
for i in range(N_raw):
    last_values = data_raw[i, -k_hours:]
    earlier = data_raw[i, :-k_hours]
    if np.all(last_values == 0.0) and np.mean(earlier) > 1.0:
        type1_indices.append(i)

# 3. 檢測類型 2: 1週滑動窗口大裂口 (Sliding Window Gap, W = 168h)
W = 168
type2_anomalies = []
for i in range(N_raw):
    if i in type1_indices:
        continue
    series = data_raw[i]
    cumsum = np.concatenate([[0.0], np.cumsum(series)])
    max_gap = -1.0
    best_t = -1
    best_m_left = 0.0
    best_m_right = 0.0
    for t in range(W, T_raw - W, 24):
        mean_left = (cumsum[t] - cumsum[t - W]) / W
        mean_right = (cumsum[t + W] - cumsum[t]) / W
        gap = abs(mean_left - mean_right)
        if gap > max_gap:
            max_gap = gap
            best_t = t
            best_m_left = mean_left
            best_m_right = mean_right
            
    if max_gap > 30.0:
        ratio_l_r = best_m_left / (best_m_right + 1e-5)
        ratio_r_l = best_m_right / (best_m_left + 1e-5)
        if ratio_l_r > 5.0 or ratio_r_l > 5.0:
            type2_anomalies.append((i, best_t, best_m_left, best_m_right, max_gap))

type2_indices = [x[0] for x in type2_anomalies]

# 4. 檢測類型 3: 稀疏脈衝型格點 (Sparse Pulse Grid)
#    條件: 零值佔比 > 30% (超過三成小時完全無人) 
#          這類格點通常為空曠山區、特殊場館等，信號太稀疏無法做有效預測
zero_ratios = (data_raw == 0.0).mean(axis=1)
type3_indices = [i for i in range(N_raw) if zero_ratios[i] > 0.30]

# 5. 檢測類型 4: 任意零值格點 (Zero-Intolerant Filter)
#    條件: 全年任意小時出現 0 值 → 視為訊號缺失，直接移除
#          我們的模型假設目標訊號為連續非零的時序，零值代表資料缺失或無人格點
type4_indices = [i for i in range(N_raw)
                 if i not in set(type1_indices + type2_indices + type3_indices)
                 and (data_raw[i] == 0.0).any()]

anomaly_indices = set(type1_indices + type2_indices + type3_indices + type4_indices)
print(f"過濾結果:")
print(f"  類型 1 (後期歸零)  = {len(type1_indices):4d} 個")
print(f"  類型 2 (突變裂口)  = {len(type2_indices):4d} 個")
print(f"  類型 3 (稀疏格點)  = {len(type3_indices):4d} 個  (零值率 > 30%)")
print(f"  類型 4 (任意零值)  = {len(type4_indices):4d} 個  (任何小時出現 0 → 移除)")
print(f"總計排除: {len(anomaly_indices)} 個異常網格，剩餘 {N_raw - len(anomaly_indices)} 個網格。")

# 排除異常數據，取得乾淨資料與座標
clean_mask = np.array([i not in anomaly_indices for i in range(N_raw)])
data_clean = data_raw[clean_mask]
coords_clean = coords[clean_mask]
grids_clean = grids_raw[clean_mask]

# 釋放原始全量 dataframe/array 記憶體，避免 Windows Paging File 溢出
del df_raw
gc.collect()


### 1.1 異常資料時序圖範例展示
我們繪製前 3 個類型 1 與前 3 個類型 2 異常資料的原始人流時序，紅色虛線表示異常發生或最大變動的位置。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Detected Anomalous Grids (Type 1: Late-Zero | Type 2: Gap | Type 3: Sparse >30% | Type 4: Spike max/mean>50)', fontsize=16, fontweight='bold')

# 繪製類型 1
for idx_plot, idx in enumerate(type1_indices[:3]):
    ax = axes[0, idx_plot]
    ax.plot(data_raw[idx], color='tab:blue', alpha=0.7)
    ax.axvline(x=T_raw - k_hours, color='red', linestyle='--', label='Start of Zero Drop-off')
    ax.set_title(f"Type 1: Grid {grids_raw[idx]}\n(last 24h = all zeros)")
    ax.set_xlabel("Hours")
    ax.set_ylabel("Population")
    ax.legend()
    ax.grid(True, alpha=0.3)

# 繪製類型 2
for idx_plot, (idx, split_t, m_left, m_right, gap) in enumerate(type2_anomalies[:3]):
    ax = axes[1, idx_plot]
    ax.plot(data_raw[idx], color='tab:orange', alpha=0.7)
    ax.axvline(x=split_t, color='red', linestyle='--', label=f'Max Gap: {gap:.1f}')
    ax.set_title(f"Type 2: Grid {grids_raw[idx]}\n(Gap at hour {split_t})")
    ax.set_xlabel("Hours")
    ax.set_ylabel("Population")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# 釋放 data_raw 記憶體
del data_raw
gc.collect()


## 2. 空間自組織根部劃分 (Spatiotemporal Root Partitioning)
在此步驟中，我們將城市劃分成 $K=5$ 個地理連續且人流行為趨勢差異最大的「根區域」：
1. **特徵提取**：計算各格點一整年 365 天的平均 **24小時日人流特徵曲線** (shape: `(N, 24)`)。
2. **標準化 (Shape-Matching)**：對每個格點的特徵曲線進行 Z-Score 標準化，僅比較「趨勢起伏形狀」，不受絕對流量大小影響。
3. **空間限制 KNN 連通圖**：建立每個格點地理上最近 8 個鄰居的連通圖 (Connectivity Graph)，確保聚類出來的區域是**地理上連續的**，而非碎裂的散點。
4. **凝聚階層聚類 (Ward Linkage)**：在空間限制下，使群體內的時序方差最小化，將城市劃分成 $K=5$ 個互斥的根部區塊。

In [ ]:
# ── 1. 先從乾淨資料中選出 NUM_GRIDS 個格點 ──
#    選擇在 data clean 步驟完成後立即確定，後續所有模型都使用同一批格點
np.random.seed(42)
N_clean = data_clean.shape[0]
selected_indices = np.random.choice(N_clean, NUM_GRIDS, replace=False)
selected_coords  = coords_clean[selected_indices]
selected_data    = data_clean[selected_indices].astype(np.float32)
selected_grids   = grids_clean[selected_indices]
print(f"已從 {N_clean} 個乾淨網格中選出 {NUM_GRIDS} 個網格進行建模。")

# ── 2. 僅對選出的 NUM_GRIDS 格點做根分區 (Spatiotemporal Root Partitioning) ──
# 提取每日平均 24 小時人流特徵曲線
daily_profiles = selected_data.reshape(NUM_GRIDS, -1, 24).mean(axis=1)  # (NUM_GRIDS, 24)

# 標準化人流特徵曲線
daily_profiles_norm = (daily_profiles - daily_profiles.mean(axis=1, keepdims=True)) / (
    daily_profiles.std(axis=1, keepdims=True) + 1e-5
)

# 建立最近鄰地理連通圖 (KNN) — 使用選出格點的座標
knn_graph = kneighbors_graph(selected_coords, n_neighbors=8, include_self=False)

# 進行空間約束凝聚層級聚類 (使用全局 K_ROOTS)
K_roots = K_ROOTS
clustering = AgglomerativeClustering(n_clusters=K_roots, connectivity=knn_graph, linkage='ward')
root_labels = clustering.fit_predict(daily_profiles_norm)

print(f"成功將 {NUM_GRIDS} 個選出網格劃分為 {K_roots} 個地理連續根區域！")
for r in range(K_roots):
    count = np.sum(root_labels == r)
    print(f"  - 區域 {r}: 網格數量 = {count} ({count/NUM_GRIDS*100:.1f}%)")


### 2.1 根部區域劃分與日通勤特徵曲線視覺化
我們繪製：
- **左圖**：格點地理分佈散佈圖，顏色代表根部區域 ID。可以看到不同區域在空間上高度連續，劃分出明晰的市中心、外圍住宅郊區、新興開發區等。
- **右圖**：展示 5 個區域的 average 24-hour 人流特徵曲線。這證明了**不同區域具有完全不同的日人流相位與規律**（例如：有的中午高、有的傍晚高、有的雙峰通勤特徵顯著），驗證了「區域特化注意力」的假設！

In [ ]:
colors = [plt.cm.tab10(i % 10) for i in range(K_roots)]

plt.figure(figsize=(16, 7))

# 1. 繪製地理分區地圖
plt.subplot(1, 2, 1)
for r in range(K_roots):
    mask = (root_labels == r)
    # FIX: Changed coords_clean to selected_coords here
    plt.scatter(selected_coords[mask, 0], selected_coords[mask, 1], 
                color=colors[r], label=f'Region {r}', s=12, alpha=0.8)
plt.title(f"Taipei Spatial Root Partition Map ({K_roots} Contiguous Clusters)", fontsize=14, fontweight='bold')
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(markerscale=3)
plt.grid(True, alpha=0.3)

# 2. 繪製 24 小時人流特徵曲線 (This part is already correct)
plt.subplot(1, 2, 2)
hours = np.arange(24)
for r in range(K_roots):
    mask = (root_labels == r)
    profile_mean = daily_profiles_norm[mask].mean(axis=0)
    profile_std = daily_profiles_norm[mask].std(axis=0)
    
    plt.plot(hours, profile_mean, color=colors[r], label=f'Region {r} Daily Profile', linewidth=2.5)
    plt.fill_between(hours, profile_mean - 0.2*profile_std, profile_mean + 0.2*profile_std, 
                     color=colors[r], alpha=0.15)

plt.title("Normalized 24-Hour Population Profiles by Region", fontsize=14, fontweight='bold')
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Normalized Population Flow (Z-score)")
plt.xticks(np.arange(0, 25, 4))
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## 3. 根區域內部的 R-Tree (MBR) 迭代建樹
在劃分好 5 個大型根區域後，我們在每個區域內部進行自底向上（Bottom-up）的幾何 R-Tree 聚合。每一個合併組都以 MBR (Minimum Bounding Rectangle) 包絡：
1. **Level 1 (微觀層)**：相鄰的 3~4 個點聚合成葉子對/三元組。
2. **Level 2 (中觀層)**：Level 1 的多個節點聚合成更大的區域 MBR。
3. **Level 3 (根節點)**：最終收斂至 5 個大型根分區。
我們在這裡實作並繪製出這套層級 MBR 的進化嵌套拓撲。

In [ ]:
class SubTreeBuilder:
    def __init__(self, region_coords, region_indices):
        self.coords = region_coords
        self.global_indices = region_indices
        
    def build_hierarchy(self):
        # 葉子節點數
        n_points = len(self.coords)
        # 初始化 Level 0 群組：每個群組只有一個點的 index
        current_groups = [[i] for i in range(n_points)]
        levels = [current_groups]
        
        # 迭代聚合 (Bottom-Up)
        while len(current_groups) > 1:
            next_groups = []
            used = set()
            n_g = len(current_groups)
            
            # 計算組與組之間的幾何中心距離
            centers = np.array([self.coords[grp].mean(axis=0) for grp in current_groups])
            
            for i in range(n_g):
                if i in used:
                    continue
                # 尋找最近且未使用的鄰居
                dists = np.linalg.norm(centers - centers[i], axis=1)
                dists[i] = np.inf
                for u in used:
                    dists[u] = np.inf
                    
                nearest = np.argmin(dists)
                if dists[nearest] < np.inf:
                    # 合併組
                    merged = current_groups[i] + current_groups[nearest]
                    next_groups.append(merged)
                    used.add(i)
                    used.add(nearest)
                else:
                    # 孤立組保留
                    next_groups.append(current_groups[i])
                    used.add(i)
            
            # 當組數不再減少時跳出
            if len(next_groups) == len(current_groups):
                break
            current_groups = next_groups
            levels.append(current_groups)
            
        return levels

# 建立森林並收集每個 Region 內部的 MBR
# root_labels 對應 selected_coords（共 NUM_GRIDS 格點）
forest_levels = {}
print("開始在每個 Region 內部建立 R-Tree 階層...")
for r in range(K_roots):
    region_mask = (root_labels == r)
    region_coords = selected_coords[region_mask]   # use selected_coords, not coords_clean
    region_indices = np.where(region_mask)[0]
    
    builder = SubTreeBuilder(region_coords, region_indices)
    levels = builder.build_hierarchy()
    forest_levels[r] = levels
    print(f"  - Region {r}: R-Tree 樹高 = {len(levels)} 層, 頂層群組數 = {len(levels[-1])}")

### 3.1 空間森林五大區域樹狀結構拓撲圖 (Hierarchical Forest Dendrograms)
由於每個區域內的格點數量龐大，直接繪製完整樹狀圖會使視覺極其混亂。因此，我們在此繪製**完整森林中所有 5 個區域（Region 0~4）的「頂層骨幹結構」**：
- 我們自頂向下展開 4 個層級。
- 最底層（葉子層）將收縮為一個標註格點數量的 summary 節點（例如 `[256 Grids]`），代表被折疊的子樹。
- 節點的連線展示了 R-Tree 的二元分群分支演化過程。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# ==================== 繪製分區樹狀結構圖 ====================
def draw_tree_structure(levels, ax, title, colors, r_color_idx):
    tree_depth = len(levels)
    plot_depth = min(4, tree_depth)
    
    nodes_to_plot = []
    
    # 1. 放入 Root 節點
    root_group = levels[-1][0]
    nodes_to_plot.append({
        'level_idx': tree_depth - 1,
        'group': root_group,
        'x': 0.5,
        'y': plot_depth,
        'parent_x': None,
        'parent_y': None,
        'label': f"Root\n({len(root_group)} Grids)"
    })
    
    # 2. 遞歸獲取子節點
    for current_plot_y in range(plot_depth - 1, 0, -1):
        parent_nodes = [n for n in nodes_to_plot if n['y'] == current_plot_y + 1]
        current_level_idx = parent_nodes[0]['level_idx'] - 1
        
        if current_level_idx < 0:
            break
        
        current_groups = levels[current_level_idx]
        
        for parent_node in parent_nodes:
            parent_group = parent_node['group']
            
            # 尋找屬於該父群組的子群組
            children = []
            for grp in current_groups:
                if set(grp).issubset(set(parent_group)) and len(grp) < len(parent_group):
                    children.append(grp)
            
            if len(children) == 0:
                continue
                
            # 二元分裂：排序以保持左右順序
            children = sorted(children, key=lambda g: len(g), reverse=True)
            
            # 動態計算水平間距，保證完美二元樹分佈，絕對不重疊
            dx = 0.5 * (0.5 ** (plot_depth - current_plot_y))
            
            if len(children) == 1:
                child = children[0]
                nodes_to_plot.append({
                    'level_idx': current_level_idx,
                    'group': child,
                    'x': parent_node['x'],
                    'y': current_plot_y,
                    'parent_x': parent_node['x'],
                    'parent_y': parent_node['y'],
                    'label': f"({len(child)} Grids)"
                })
            elif len(children) >= 2:
                # 僅顯示前兩個最大的分支
                left_child  = children[0]
                right_child = children[1]
                
                nodes_to_plot.append({
                    'level_idx': current_level_idx,
                    'group': left_child,
                    'x': parent_node['x'] - dx,
                    'y': current_plot_y,
                    'parent_x': parent_node['x'],
                    'parent_y': parent_node['y'],
                    'label': f"({len(left_child)} Grids)"
                })
                nodes_to_plot.append({
                    'level_idx': current_level_idx,
                    'group': right_child,
                    'x': parent_node['x'] + dx,
                    'y': current_plot_y,
                    'parent_x': parent_node['x'],
                    'parent_y': parent_node['y'],
                    'label': f"({len(right_child)} Grids)"
                })
                
    # 3. 開始繪製
    for node in nodes_to_plot:
        # 繪製連線 (加粗且顏色柔和)
        if node['parent_x'] is not None:
            ax.plot([node['x'], node['parent_x']], [node['y'], node['parent_y']], 
                    color='#999999', linestyle='-', linewidth=3.0, zorder=1)
            
        # 繪製節點文字與邊框
        is_leaf     = (node['y'] == 1)
        edge_color  = colors[r_color_idx]
        
        # 再次加大字體以提高可讀性
        fs = 18 if not is_leaf else 15 
        
        ax.text(node['x'], node['y'], node['label'],
                ha='center', va='center',
                fontsize=fs, fontweight='bold',
                color='#111111',
                fontfamily='Times New Roman',
                zorder=2,
                bbox=dict(boxstyle="round,pad=0.7", 
                          fc="white" if not is_leaf else "#F9F9F9", 
                          ec=edge_color,
                          lw=3.0, 
                          alpha=1.0))
        
    # 收緊座標軸以縮短視覺上的連接線長度
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(0.5, plot_depth + 0.5)
    
    # 設定大字體標題
    ax.set_title(title, fontsize=20, fontweight='bold', color='black',
                 fontfamily='Times New Roman', pad=15)
                 
    # ===== 加入分區切線（保留邊框但隱藏座標軸） =====
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#BBBBBB')       # 灰色分隔線
        spine.set_linestyle('--')        # 虛線樣式
        spine.set_linewidth(2.0)         # 邊框粗細
    ax.set_facecolor('#FCFCFC')          # 加上淡淡的背景色增加區隔感
# ── Dynamic subplots grid according to K_roots ──
# 調整邏輯：如果剛好是 4 個，就用 2x2 排列；否則最多 3 欄
if K_roots == 4:
    n_cols = 2
else:
    n_cols = min(K_roots, 3)

n_rows = math.ceil(K_roots / n_cols)

# 畫布再放大，給予充足的顯示空間 (依照你調整的尺寸)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12 * n_cols, 6 * n_rows))

# 處理只有 1 個子圖或多維度陣列的情況
if n_rows == 1 and n_cols == 1:
    axes_flat = [axes]
else:
    axes_flat = np.array(axes).flatten()

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple',
          'tab:brown', 'tab:pink', 'tab:gray']

for r in range(K_roots):
    ax = axes_flat[r]
    levels = forest_levels[r]
    draw_tree_structure(levels, ax, f"Region {r} Dendrogram", colors, r % len(colors))

# Remove unused axes (當 K_roots 不是 n_rows * n_cols 的倍數時)
for r in range(K_roots, len(axes_flat)):
    fig.delaxes(axes_flat[r])

plt.suptitle("Hierarchical R-Tree Structures for All Regions",
             fontsize=26, fontweight='bold', y=0.97,
             fontfamily='Times New Roman')
             
# 調整子圖間的間距，確保切線清晰且不擁擠
plt.tight_layout(rect=[0, 0, 1, 0.95], w_pad=4.0, h_pad=4.0)
plt.show()

## 4. 深度學習模型實驗與基準對比 (Deep Learning Models Benchmark)
本節我們在篩選後的 500 個網格上，對比以下 SOTA 時序基準模型與本研究 proposed 架構：
1. **Each-Grid LSTM (獨立 LSTM)**：為每個網格獨立配置一個 LSTM 參數（向量化 GPU 並行優化實作）。
2. **All-Grid LSTM (共享 LSTM)**：所有網格共用一個單一 LSTM 模型。
3. **DLinear (AAAI 2023 LTSF-Linear)**：趨勢 + 殘差分解雙線性層，極簡但表現強勁的 Linear 基準。
4. **PatchTST (ICLR 2023)**：Channel-Independent Patch Transformer SOTA。
5. **iTransformer (no mask) (ICLR 2024)**：將 Variate (格點) 作為 Token 的 Backbone（全連接注意力，無空間 Mask）。
6. **Proposed HMST-v2 (Full)**：層級掩碼空間 Transformer，結合 iTransformer backbone + RevIN + R-Tree 階層空間遮罩。


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
import time as _time_module
from torch.utils.data import DataLoader

# ── Import everything from the hmst package ──────────────────────────────────
from hmst.model  import (HMSTv2, TimeEmbedding, HierarchicalBlock,
                          RevIN, EachGridLSTM, AllGridLSTM, AllGridConvLSTM,
                          AllGridDLinear, AllGridPatchTST)
from hmst.train  import CHTDataset, make_loaders, run_training
from hmst.utils  import MODEL_SIZES, TRAIN_CFG, calculate_metrics, build_forest_masks
from hmst.utils  import SMALL, BASE, LARGE

# ─── Train / Val / Test split ────────────────────────────────────────────────
T_total     = selected_data.shape[1]
split_train = int(T_total * 0.70)
split_val   = int(T_total * 0.85)

train_raw = selected_data[:, :split_train]
val_raw   = selected_data[:, split_train:split_val]
test_raw  = selected_data[:, split_val:]

# ─── Build DataLoaders ───────────────────────────────────────────────────────
train_loader, val_loader, test_loader = make_loaders(
    selected_data,
    lookback   = LOOKBACK,
    batch_size = 32,
    train_frac = 0.70,
    val_frac   = 0.85,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Dataset ready ({NUM_GRIDS} grids, LOOKBACK={LOOKBACK}):")
print(f"  Train: {train_raw.shape[1]}h  |  Val: {val_raw.shape[1]}h  |  Test: {test_raw.shape[1]}h")


In [ ]:
# =============================================================================
# Build Hierarchical R-Tree Attention Masks (HMST prerequisite)
# =============================================================================
# The four masks encode spatial attention scope at each R-Tree level:
#   forest_masks[0] — Level 0: self-loop (identity)
#   forest_masks[1] — Level 1: micro-cluster attention
#   forest_masks[2] — Level 2: mid-region attention
#   forest_masks[3] — Level 3: root-region attention

from hmst.utils import build_forest_masks

HMST_MASKS = build_forest_masks(
    num_grids    = NUM_GRIDS,
    K_roots      = K_roots,
    root_labels  = root_labels,
    forest_levels= forest_levels,
)
print(f"Built {len(HMST_MASKS)} hierarchical attention masks — each shape: {HMST_MASKS[0].shape}")


In [ ]:
# =============================================================================
# § 4  Benchmark Models — Small / Base / Large
# =============================================================================
# MODEL_SIZES ensures the same hidden_dim / d_model for a given size tier
# across ALL model types.  TRAIN_CFG is shared by every model.
#
#   Size    hidden_dim / d_model   d_k   num_layers   nhead
#   small        16                  8        1          2
#   base         32                 16        2          4
#   large        64                 32        4          8
# =============================================================================

benchmark_models = {}

# ── Each-Grid LSTM (Small / Base / Large) ────────────────────────────────────
for sz in ("small", "base", "large"):
    benchmark_models[f"Each-Grid LSTM ({sz.capitalize()})"] = EachGridLSTM(
        num_grids  = NUM_GRIDS,
        lookback   = LOOKBACK,
        hidden_dim = MODEL_SIZES[sz]["hidden_dim"],
    )

# ── All-Grid LSTM (Small / Base / Large) ────────────────────────────────────
for sz in ("small", "base", "large"):
    benchmark_models[f"All-Grid LSTM ({sz.capitalize()})"] = AllGridLSTM(
        hidden_dim = MODEL_SIZES[sz]["hidden_dim"],
    )

# ── All-Grid ConvLSTM (Small / Base / Large) ─────────────────────────────────
for sz in ("small", "base", "large"):
    benchmark_models[f"All-Grid ConvLSTM ({sz.capitalize()})"] = AllGridConvLSTM(
        hidden_dim = MODEL_SIZES[sz]["hidden_dim"],
    )

# ── All-Grid DLinear (parameter-free; no size variant) ───────────────────────
benchmark_models["All-Grid DLinear"] = AllGridDLinear(
    lookback    = LOOKBACK,
    kernel_size = 3,
)

# ── All-Grid PatchTST (Base / Large) ─────────────────────────────────────────
for sz in ("base", "large"):
    s = MODEL_SIZES[sz]
    benchmark_models[f"All-Grid PatchTST ({sz.capitalize()})"] = AllGridPatchTST(
        lookback   = LOOKBACK,
        patch_len  = 4,
        stride     = 4,
        d_model    = s["d_model"],
        nhead      = s["nhead"],
        num_layers = s["num_layers"],
    )

# ── HMST-v2 (Base / Large) ───────────────────────────────────────────────────
for sz in ("base", "large"):
    s = MODEL_SIZES[sz]
    benchmark_models[f"HMST-v2 ({sz.capitalize()})"] = HMSTv2(
        num_grids      = NUM_GRIDS,
        lookback       = LOOKBACK,
        d_model        = s["d_model"],
        d_k            = s["d_k"],
        num_layers     = s["num_layers"],
        dropout        = 0.1,
        forest_masks   = HMST_MASKS,
        use_revin      = True,
        use_rtree_mask = True,
        use_spatial_emb= True,
        use_time_emb   = True,
    )

# ── Print parameter count summary ────────────────────────────────────────────
print(f"{'Model':<42} {'Params (K)':>10}")
print("-" * 54)
for name, m in benchmark_models.items():
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {name:<40} {n/1000:>9.1f}K")


In [ ]:
# =============================================================================
# Metrics: MAE, RMSE, wMAPE, Daily DTW, Step DTW
# =============================================================================
import copy

def fast_dtw_distance(s1, s2, w=24):
    T = len(s1)
    dtw_mat = np.full((T+1, T+1), np.inf)
    dtw_mat[0, 0] = 0.0
    for i in range(1, T+1):
        for j in range(max(1, i-w), min(T+1, i+w+1)):
            cost = abs(s1[i-1] - s2[j-1])
            dtw_mat[i, j] = cost + min(dtw_mat[i-1,j], dtw_mat[i,j-1], dtw_mat[i-1,j-1])
    return dtw_mat[T, T]

def calculate_metrics(y_true, y_pred, dtw_grids=50):
    """
    y_true, y_pred : (T, N)  raw scale
    Returns: mae, rmse, wmape(%), daily_dtw, step_dtw
      daily_dtw : average DTW over 24h day segments
      step_dtw  : full-sequence DTW / T  (normalised per-step)
    """
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    wmape = np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100

    T, N = y_true.shape
    sel  = np.random.choice(N, min(dtw_grids, N), replace=False)

    # ── Daily DTW (24h segments) ──
    n_days = T // 24
    daily_dtws = []
    for d in range(n_days):
        for n in sel:
            daily_dtws.append(
                fast_dtw_distance(y_true[d*24:(d+1)*24, n],
                                  y_pred[d*24:(d+1)*24, n], w=6))
    daily_dtw = np.mean(daily_dtws) if daily_dtws else np.nan

    # ── Step DTW (full sequence, normalised by T) ──
    step_dtws = []
    for n in sel:
        step_dtws.append(fast_dtw_distance(y_true[:, n], y_pred[:, n], w=24) / T)
    step_dtw = np.mean(step_dtws)

    return mae, rmse, wmape, daily_dtw, step_dtw


In [ ]:
# =============================================================================
# § 4  Training Loop — run all benchmark models
# =============================================================================
results     = {}
preds_store = {}

for name, model in benchmark_models.items():
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    mdl, preds, trues, pe, te, t_secs, n_params = run_training(
        name         = name,
        model        = model,
        cfg          = TRAIN_CFG,
        train_loader = train_loader,
        val_loader   = val_loader,
        test_loader  = test_loader,
        device       = device,
    )
    mae, rmse, wmape, ddtw, sdtw = calculate_metrics(te, pe)
    results[name] = {
        "MAE":        mae,
        "RMSE":       rmse,
        "wMAPE (%)":  wmape,
        "Daily DTW":  ddtw,
        "Step DTW":   sdtw,
        "Params (K)": round(n_params / 1000, 1),
        "Train (s)":  round(t_secs, 1),
    }
    preds_store[name] = (preds, trues)
    print(f"  TEST  MAE={mae:.4f}  wMAPE={wmape:.2f}%  Params={n_params/1e3:.1f}K  Time={t_secs:.1f}s")

# Keep references to the Large HMST for trajectory plots
hmst_model_trained = benchmark_models["HMST-v2 (Large)"]
hmst_preds, hmst_trues = preds_store["HMST-v2 (Large)"]
print("\nBenchmark complete.")


In [ ]:
# =============================================================================
# Benchmark Results Table
# =============================================================================
df_bench = pd.DataFrame(results).T
print("\n" + "="*50 + " BENCHMARK RESULTS " + "="*50)
print(df_bench.to_markdown(floatfmt=".4f"))
print("="*119)

# Compute per-grid MAE for each model (for trajectory selection)
per_grid_mae = {}
for name, (p, t) in preds_store.items():
    per_grid_mae[name] = np.mean(np.abs(p - t), axis=0)   # (N,)

# Best & worst baseline (by overall MAE, excluding HMST)
bl_names  = [n for n in results if "HMST" not in n]
best_bl   = min(bl_names, key=lambda n: results[n]["MAE"])
worst_bl  = max(bl_names, key=lambda n: results[n]["MAE"])
print(f"\nBest  baseline: {best_bl}")
print(f"Worst baseline: {worst_bl}")

# HMST advantage score per grid
hmst_mae    = per_grid_mae["HMST-v2 (Base)"]
best_bl_mae = per_grid_mae[best_bl]
adv_score   = best_bl_mae - hmst_mae   # + means HMST wins

case_a_grid = int(np.argmax(adv_score))    # HMST wins most
case_b_grid = int(np.argmin(adv_score))    # HMST loses most
print(f"\nCase A (HMST best advantage): Grid {case_a_grid}")
print(f"Case B (HMST worst):            Grid {case_b_grid}")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# =============================================================================
# Trade-off Visualization (Params vs Accuracy & Time vs Accuracy)
# =============================================================================

# Colour + shape registry per model family
FAMILY_STYLE = {
    "HMST":            {"color": "#c0392b", "marker": "o", "s": 180, "zorder": 4},
    "PatchTST":        {"color": "#2980b9", "marker": "o", "s": 140, "zorder": 3},
    "Each-Grid LSTM":  {"color": "#27ae60", "marker": "D", "s": 100, "zorder": 3},
    "All-Grid LSTM":   {"color": "#8e6914", "marker": "o", "s": 120, "zorder": 3},
    "ConvLSTM":        {"color": "#6c5ce7", "marker": "o", "s": 120, "zorder": 3},
    "DLinear":         {"color": "#636e72", "marker": "o", "s": 140, "zorder": 3},
}

def _family(name):
    if "HMST"          in name: return "HMST"
    if "PatchTST"      in name: return "PatchTST"
    if "Each-Grid"     in name: return "Each-Grid LSTM"
    if "ConvLSTM"      in name: return "ConvLSTM"
    if "All-Grid LSTM" in name: return "All-Grid LSTM"
    return "DLinear"

SIZE_ORDER = {"Small": 0, "Base": 1, "Large": 2, "": 0}

def _size(name):
    for tok in ("Small", "Base", "Large"):
        if tok in name:
            return tok
    return ""

# Per-point label offsets (manual nudge to avoid overlap)
LABEL_OFFSETS = {
    "Each-Grid LSTM (Small)":  ( 10,  14),
    "Each-Grid LSTM (Base)":   ( 10,   0),
    "Each-Grid LSTM (Large)":  ( 10, -14),
    "All-Grid LSTM (Small)":   (-10,  14),
    "All-Grid LSTM (Base)":    (-10,   0),
    "All-Grid LSTM (Large)":   (-10, -14),
    "ConvLSTM (Small)":        (-10,  14),
    "ConvLSTM (Base)":         (-10,   0),
    "ConvLSTM (Large)":        (-10, -14),
    "DLinear":                 (-10,   6),
    "PatchTST (Base)":         ( 10,   8),
    "PatchTST (Large)":        (-10,  -8),
    "HMST-v2 (Base)":          ( 10,   8),
    "HMST-v2 (Large)":         ( 10,  -8),
}

def _offset(name):
    for key, off in LABEL_OFFSETS.items():
        if key in name:
            return off
    return (10, 5)

def _ha(name):
    ox, _ = _offset(name)
    return "left" if ox >= 0 else "right"

def _display(name):
    return (name
            .replace("All-Grid ",  "All-Grid\n")
            .replace("Each-Grid ", "Each-Grid\n")
            .replace("HMST-v2 ",   "HMST-v2\n"))

def plot_tradeoff(ax, x_metric, y_metric, x_label, y_label, title):
    # Group points by family for dashed connector lines
    families = {}
    for name, row in df_bench.iterrows():
        fam = _family(name)
        sz  = _size(name)
        x, y = float(row[x_metric]), float(row[y_metric])
        families.setdefault(fam, []).append((SIZE_ORDER[sz], x, y, name))

    # Draw dashed connectors between sizes within the same family
    for fam, pts in families.items():
        if len(pts) < 2:
            continue
        pts_sorted = sorted(pts, key=lambda t: t[0])   # Small -> Base -> Large
        xs = [p[1] for p in pts_sorted]
        ys = [p[2] for p in pts_sorted]
        sty = FAMILY_STYLE[fam]
        ax.plot(xs, ys, ls="--", lw=1.3, color=sty["color"],
                alpha=0.55, zorder=sty["zorder"] - 1)

    # Draw scatter points and labels — all text in black
    for name, row in df_bench.iterrows():
        x, y = float(row[x_metric]), float(row[y_metric])
        fam  = _family(name)
        sty  = FAMILY_STYLE[fam]
        ox, oy = _offset(name)

        ax.scatter(x, y,
                   c=sty["color"], s=sty["s"], marker=sty["marker"],
                   edgecolors="black", linewidths=0.8,
                   alpha=0.90, zorder=sty["zorder"])

        ax.annotate(
            _display(name),
            (x, y),
            xytext=(ox, oy),
            textcoords="offset points",
            fontsize=9.5,
            fontfamily="Times New Roman",
            fontweight="normal",
            color="black",
            zorder=5,
            va="center",
            ha=_ha(name),
        )

    ax.set_title(title, fontsize=14, fontfamily="Times New Roman", color="black")
    ax.set_xlabel(x_label, fontsize=12, fontfamily="Times New Roman", color="black")
    ax.set_ylabel(y_label, fontsize=12, fontfamily="Times New Roman", color="black")
    ax.tick_params(colors="black")
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
    ax.grid(True, alpha=0.25, color="gray")
    ax.set_facecolor("#FCFCFC")
    ax.margins(x=0.18, y=0.18)


fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
fig.suptitle("Model Performance Trade-offs (Lower-Left is Better)",
             fontsize=16, fontfamily="Times New Roman", color="black")

# Legend patches
legend_handles = [
    mpatches.Patch(color=v["color"], label=k)
    for k, v in FAMILY_STYLE.items()
]
legend_handles.append(
    plt.Line2D([0], [0], ls="--", color="gray", lw=1.3,
               label="Size: Small \u2192 Base \u2192 Large")
)

# Subplot 1: Params vs MAE
plot_tradeoff(axes[0],
              x_metric="Params (K)", y_metric="MAE",
              x_label="Model Complexity (Params in K)",
              y_label="MAE (Accuracy)",
              title="Complexity vs. Accuracy (MAE)")
axes[0].legend(handles=legend_handles, fontsize=8.5,
               framealpha=0.85, edgecolor="black",
               prop={"family": "Times New Roman"})

# Subplot 2: Train Time vs wMAPE
plot_tradeoff(axes[1],
              x_metric="Train (s)", y_metric="wMAPE (%)",
              x_label="Training Efficiency (Seconds)",
              y_label="wMAPE (%)",
              title="Speed vs. Accuracy (wMAPE)")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# ── Trajectory: Case A — HMST Best, All Baselines ──
T_SHOW = 168
gid    = case_a_grid

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hmst_trues[:T_SHOW, gid], color="black", lw=2.5, label="Ground Truth", alpha=0.9)
ax.plot(preds_store["HMST-v2 (Large)"][0][:T_SHOW, gid],
        color="#c0392b", lw=2, ls="-", label="HMST-v2 (Proposed)", zorder=5)

bl_colors = {
    "Each-Grid LSTM": "#3498db",
    "All-Grid LSTM": "#2980b9",
    "DLinear": "#e67e22",
    "PatchTST": "#27ae60",
    "iTransformer (no mask)": "#8e44ad"
}
for bname, bcol in bl_colors.items():
    if bname in preds_store:
        ax.plot(preds_store[bname][0][:T_SHOW, gid],
                color=bcol, lw=1.4, ls="--", label=bname, alpha=0.80)

ax.set_title(f"Case A — Grid {gid}: HMST-v2 Advantages (HMST MAE={hmst_mae[gid]:.1f}, "
             f"Best Baseline MAE={best_bl_mae[gid]:.1f})")
ax.set_xlabel("Hours"); ax.set_ylabel("Population Flow")
ax.legend(loc="upper right", framealpha=0.85)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Trajectory: Case B — HMST Worst, All Baselines ──
T_SHOW = 168
gid    = case_b_grid

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hmst_trues[:T_SHOW, gid], color="black", lw=2.5, label="Ground Truth", alpha=0.9)
ax.plot(preds_store["HMST-v2 (Large)"][0][:T_SHOW, gid],
        color="#c0392b", lw=2, ls="-", label="HMST-v2 (Proposed)", zorder=5)

bl_colors = {
    "Each-Grid LSTM": "#3498db",
    "All-Grid LSTM": "#2980b9",
    "DLinear": "#e67e22",
    "PatchTST": "#27ae60",
    "iTransformer (no mask)": "#8e44ad"
}
for bname, bcol in bl_colors.items():
    if bname in preds_store:
        ax.plot(preds_store[bname][0][:T_SHOW, gid],
                color=bcol, lw=1.4, ls="--", label=bname, alpha=0.80)

ax.set_title(f"Case B — Grid {gid}: HMST-v2 Disadvantages (HMST MAE={hmst_mae[gid]:.1f}, "
             f"Best Baseline MAE={best_bl_mae[gid]:.1f})")
ax.set_xlabel("Hours"); ax.set_ylabel("Population Flow")
ax.legend(loc="upper right", framealpha=0.85)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 5. Ablation Study

我們設計消融實驗以評估 HMST-v2 各核心組件的貢獻：
- **Full (Proposed)**：完整模型，啟用 RevIN, 空間與時間特徵嵌入, 還有 R-Tree 階層 Masked 多頭注意力。
- **w/o RevIN**：停用 RevIN（可逆實例歸一化），測試模型對抗分佈漂移與極端數據變動時的表現。
- **w/o Time Embedding**：停用 Hour 與 Day-of-Week 時間上下文嵌入，評估時間週期特徵的貢獻。
- **w/o Spatial Position**：停用空間坐標 Position Embedding，評估靜態空間特徵的貢獻。

| Variant | RevIN | 空間嵌入 | 時間嵌入 | R-Tree Mask | 說明 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **Full** | ✅ | ✅ | ✅ | ✅ | 完整時空階層模型 |
| **w/o RevIN** | ❌ | ✅ | ✅ | ✅ | 無分佈校準 |
| **w/o Time Embedding** | ✅ | ✅ | ❌ | ✅ | 無時間週期特徵 |
| **w/o Spatial Position** | ✅ | ❌ | ✅ | ✅ | 無空間位置特徵 |


In [ ]:
# =============================================================================
# § 5  Ablation Study — Full / w/o RevIN / w/o Time Emb / w/o Spatial Pos
# =============================================================================
# All ablation variants use the LARGE size for a fair, high-capacity comparison.
# Training config is shared (TRAIN_CFG).

_abl_size = LARGE   # alias: MODEL_SIZES["large"]

ablation_variants = {
    "Full": HMSTv2(
        NUM_GRIDS, LOOKBACK,
        d_model=_abl_size["d_model"], d_k=_abl_size["d_k"],
        num_layers=_abl_size["num_layers"], dropout=0.1,
        forest_masks=HMST_MASKS,
        use_revin=True,  use_rtree_mask=True,  random_mask=False,
        use_spatial_emb=True,  use_time_emb=True,
    ),
    "w/o RevIN": HMSTv2(
        NUM_GRIDS, LOOKBACK,
        d_model=_abl_size["d_model"], d_k=_abl_size["d_k"],
        num_layers=_abl_size["num_layers"], dropout=0.1,
        forest_masks=HMST_MASKS,
        use_revin=False, use_rtree_mask=True,  random_mask=False,
        use_spatial_emb=True,  use_time_emb=True,
    ),
    "w/o Time Embedding": HMSTv2(
        NUM_GRIDS, LOOKBACK,
        d_model=_abl_size["d_model"], d_k=_abl_size["d_k"],
        num_layers=_abl_size["num_layers"], dropout=0.1,
        forest_masks=HMST_MASKS,
        use_revin=True,  use_rtree_mask=True,  random_mask=False,
        use_spatial_emb=True,  use_time_emb=False,
    ),
    "w/o Spatial Position": HMSTv2(
        NUM_GRIDS, LOOKBACK,
        d_model=_abl_size["d_model"], d_k=_abl_size["d_k"],
        num_layers=_abl_size["num_layers"], dropout=0.1,
        forest_masks=HMST_MASKS,
        use_revin=True,  use_rtree_mask=True,  random_mask=False,
        use_spatial_emb=False, use_time_emb=True,
    ),
    "w/o R-Tree Mask": HMSTv2(
        NUM_GRIDS, LOOKBACK,
        d_model=_abl_size["d_model"], d_k=_abl_size["d_k"],
        num_layers=_abl_size["num_layers"], dropout=0.1,
        forest_masks=HMST_MASKS,
        use_revin=True,  use_rtree_mask=False, random_mask=False,
        use_spatial_emb=True,  use_time_emb=True,
    ),
    "Random Mask": HMSTv2(
        NUM_GRIDS, LOOKBACK,
        d_model=_abl_size["d_model"], d_k=_abl_size["d_k"],
        num_layers=_abl_size["num_layers"], dropout=0.1,
        forest_masks=HMST_MASKS,
        use_revin=True,  use_rtree_mask=True,  random_mask=True,
        use_spatial_emb=True,  use_time_emb=True,
    ),
}

abl_results  = {}
abl_preds_map = {}

for var, model in ablation_variants.items():
    print(f"\n{'='*50}\n  Ablation: {var}\n{'='*50}")
    _, preds, trues, pe, te, t_secs, n_params = run_training(
        var, model, TRAIN_CFG, train_loader, val_loader, test_loader, device
    )
    mae, rmse, wmape, ddtw, sdtw = calculate_metrics(te, pe)
    abl_results[var]    = {"MAE": mae, "RMSE": rmse, "wMAPE (%)": wmape,
                            "Daily DTW": ddtw, "Step DTW": sdtw,
                            "Params (K)": n_params / 1000, "Train (s)": t_secs}
    abl_preds_map[var]  = (preds, trues)
    print(f"  MAE={mae:.4f} | {n_params/1e3:.1f}K params | {t_secs:.1f}s")

df_abl = pd.DataFrame(abl_results).T
print("\n" + "="*30 + " ABLATION RESULTS " + "="*30)
print(df_abl.to_markdown(floatfmt=".4f"))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6', '#f39c12', '#1abc9c']
for ax, metric in zip(axes, ["MAE", "Train (s)"]):
    vals = [abl_results[v][metric] for v in abl_results]
    bars = ax.bar(list(abl_results.keys()), vals, color=palette[:len(vals)],
                  edgecolor='black', lw=0.8)
    ax.set_title(f"Ablation: {metric}", fontsize=13, fontweight='bold', color='black')
    ax.set_ylabel(metric, color='black')
    ax.tick_params(colors='black')
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()*1.01,
                f'{v:.2f}', ha='center', va='bottom', fontsize=9, color='black')
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## 6. Region Swap Adaptation Experiment

**Cascade 設計**：按格點數量將 5 個根區域排序（A 最大），讓 B 的模式替代 C，A 的模式替代 B。  
只評估被改動的 B+C 格點。展示 6 張視覺化（F1~F6）。


In [ ]:
# =============================================================================
# Swap Setup: Dynamic Cascade based on K_ROOTS
# =============================================================================
np.random.seed(42)

# 找出各 Region 在 selected_indices 中的格點
region_sel = {}
for r in range(K_ROOTS):
    # Change root_labels[gi] to root_labels[si]
    region_sel[r] = [si for si, gi in enumerate(selected_indices)
                     if root_labels[si] == r]
    print(f"  Region {r}: {len(region_sel[r])} grids in selected_indices")

# 按大小排序 (大→小)
sorted_r = sorted(region_sel.keys(), key=lambda r: -len(region_sel[r]))
A_idx = sorted_r[0]
B_idx = sorted_r[1] if len(sorted_r) > 1 else sorted_r[0]
C_idx = sorted_r[2] if len(sorted_r) > 2 else B_idx

grids_A = region_sel[A_idx]
grids_B = region_sel[B_idx]
grids_C = region_sel[C_idx]

pct_changed = (len(grids_B) + len(grids_C)) / NUM_GRIDS * 100
print(f"\nCascade: A={A_idx}({len(grids_A)}) → B={B_idx}({len(grids_B)}) → C={C_idx}({len(grids_C)})")
print(f"Changed: {len(grids_B)+len(grids_C)}/500 = {pct_changed:.1f}% of all grids")

# ── Cascade replace ──
adapted_data = selected_data.copy()

# Step 1: C ← B pattern
b_samp = np.random.choice(len(grids_B), len(grids_C), replace=(len(grids_B)<len(grids_C)))
orig_C_data = selected_data[grids_C].copy()
for i, c in enumerate(grids_C):
    adapted_data[c] = selected_data[grids_B[b_samp[i]]]

# Step 2: B ← A pattern
a_samp = np.random.choice(len(grids_A), len(grids_B), replace=(len(grids_A)<len(grids_B)))
orig_B_data = selected_data[grids_B].copy()
for i, b in enumerate(grids_B):
    adapted_data[b] = selected_data[grids_A[a_samp[i]]]

# ── F4: Difference Coefficient ──
def diff_coeff(before, after):
    bp = before.reshape(-1, 24).mean(axis=0)
    ap = after.reshape(-1, 24).mean(axis=0)
    c  = np.corrcoef(bp, ap)[0, 1]
    return 1 - c

dc_B = [diff_coeff(selected_data[b], adapted_data[b]) for b in grids_B]
dc_C = [diff_coeff(selected_data[c], adapted_data[c]) for c in grids_C]
print(f"\nDiff coefficient — B: mean={np.mean(dc_B):.3f} | C: mean={np.mean(dc_C):.3f}")

# ── F1: Swap map ──
fig, ax = plt.subplots(figsize=(9, 7))
region_color = {A_idx: '#3498db', B_idx: '#e74c3c', C_idx: '#f39c12'}
for r in range(K_ROOTS):
    # Vectorized NumPy comparison fixes the index out of bounds error
    mask_r = (root_labels == r)
    
    color  = region_color.get(r, '#bdc3c7')
    label  = {A_idx: f'A (unchanged, {len(grids_A)} grids)',
               B_idx: f'B → A pattern ({len(grids_B)} grids)',
               C_idx: f'C → B pattern ({len(grids_C)} grids)'}.get(r, f'Region {r} (unchanged)')
    
    ax.scatter(selected_coords[mask_r, 0], selected_coords[mask_r, 1],
               color=color, label=label, s=15, alpha=0.8)

ax.set_title(f"F1: Cascade Swap Map ({pct_changed:.1f}% of grids changed)",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.legend(markerscale=2); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── F2: Before/After time series (3 from B, 3 from C) ──
n_ex = 3
fig, axes = plt.subplots(2, n_ex, figsize=(18, 8))
for ax_row, (grids, orig, region_label, color) in enumerate([
        (grids_B[:n_ex], orig_B_data, f'Region B ({A_idx}→{B_idx})', '#e74c3c'),
        (grids_C[:n_ex], orig_C_data, f'Region C ({B_idx}→{C_idx})', '#f39c12')]):
    for j in range(n_ex):
        ax = axes[ax_row, j]
        T_show = min(168, selected_data.shape[1])
        ax.plot(orig[j, :T_show], color='gray', ls='--', lw=1.5, label='Before', alpha=0.8)
        ax.plot(adapted_data[grids[j], :T_show], color=color, lw=1.5, label='After')
        ax.set_title(f"{region_label}\nGrid {grids[j]}", fontsize=10)
        ax.set_xlabel("Hours"); ax.set_ylabel("Population")
        if j == 0: ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
plt.suptitle("F2: Before vs After Swap (first 168h)", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# ── F3: 24h Daily Profile ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hours_x = np.arange(24)
for ax, (grids, orig, label, color) in zip(axes, [
        (grids_B, orig_B_data, f'Region B (A→B swap)', '#e74c3c'),
        (grids_C, orig_C_data, f'Region C (B→C swap)', '#f39c12')]):
    before_prof = orig.reshape(len(grids), -1, 24).mean(axis=(0,1))
    after_prof  = adapted_data[grids].reshape(len(grids), -1, 24).mean(axis=(0,1))
    ax.plot(hours_x, before_prof, color='gray',  ls='--', lw=2, label='Before')
    ax.plot(hours_x, after_prof,  color=color,            lw=2, label='After')
    ax.fill_between(hours_x, before_prof, after_prof, alpha=0.15, color=color)
    ax.set_title(f"F3: 24h Daily Profile — {label}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Hour of Day"); ax.set_ylabel("Avg Population")
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── F4: Difference Coefficient Violin ──
fig, ax = plt.subplots(figsize=(7, 5))
ax.violinplot([dc_B, dc_C], positions=[1, 2], showmedians=True)
ax.set_xticks([1, 2])
ax.set_xticklabels([f'Region B\n(A→B, n={len(dc_B)})',
                    f'Region C\n(B→C, n={len(dc_C)})'])
ax.axhline(np.mean(dc_B), color='#e74c3c', ls='--', alpha=0.7,
           label=f'B mean={np.mean(dc_B):.3f}')
ax.axhline(np.mean(dc_C), color='#f39c12', ls='--', alpha=0.7,
           label=f'C mean={np.mean(dc_C):.3f}')
ax.set_title("F4: Distribution of Difference Coefficients\n"
             "(1-Pearson, 0=identical, 2=opposite)",
             fontsize=12, fontweight='bold')
ax.set_ylabel("Diff Coeff (1 - Pearson Corr)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# Adaptation Strategies (Zero-Shot / HMST Fine-Tune / Baseline Retrain)
# =============================================================================
import copy as _copy

adapted_train = adapted_data[:, :split_train]
adapted_val   = adapted_data[:, split_train:split_val]
adapted_test  = adapted_data[:, split_val:]

bc_indices = np.array(grids_B + grids_C)
bc_mask    = bc_indices

ad_train_ds = CHTDataset(adapted_train, LOOKBACK, 0)
ad_val_ds   = CHTDataset(adapted_val,   LOOKBACK, split_train)
ad_test_ds  = CHTDataset(adapted_test,  LOOKBACK, split_val)
ad_train_l  = DataLoader(ad_train_ds, 32, shuffle=True,  drop_last=False)
ad_val_l    = DataLoader(ad_val_ds,   32, shuffle=False, drop_last=False)
ad_test_l   = DataLoader(ad_test_ds,  32, shuffle=False, drop_last=False)

adaptation_results = {}
adapt_preds_store  = {}
bc_tensor = torch.tensor(bc_indices, dtype=torch.long)

def collect_preds(model, loader, device, grid_mask=None):
    model.eval()
    ps, ts = [], []
    with torch.no_grad():
        for xb, yb, hb, db in loader:
            ps.append(model(xb.to(device), hb.to(device), db.to(device)).cpu().numpy())
            ts.append(yb.numpy())
    p = np.concatenate(ps, axis=0)[:, :, 0]
    t = np.concatenate(ts, axis=0)[:, :, 0]
    if grid_mask is not None:
        return p[:, grid_mask], t[:, grid_mask], p, t
    return p, t, p, t

# ── 1. Evaluate ALL Pre-trained Models directly (Zero-Shot) on the swapped test data ──
print("\n--- Zero-Shot evaluations on swapped test data ---")

# HMST Zero-Shot
zs_pe, zs_te, zs_p, zs_t = collect_preds(hmst_model_trained, ad_test_l, device, bc_mask)
mae, *_ = calculate_metrics(zs_te, zs_pe)
adaptation_results["HMST Zero-Shot"]  = {"MAE": mae, "Time (s)": 0.0}
adapt_preds_store["HMST Zero-Shot"]   = (zs_p, zs_t)
print(f"  HMST Zero-Shot MAE on B+C = {mae:.4f}")

# Baselines Zero-Shot
zero_shot_baselines = [
    "Each-Grid LSTM", "All-Grid LSTM", "All-Grid ConvLSTM", 
    "All-Grid DLinear", "All-Grid PatchTST (Large)"
]
for bname in zero_shot_baselines:
    bmodel = benchmark_models[bname]
    b_pe, b_te, b_p, b_t = collect_preds(bmodel, ad_test_l, device, bc_mask)
    mae, *_ = calculate_metrics(b_te, b_pe)
    adaptation_results[f"{bname}\nZero-Shot"] = {"MAE": mae, "Time (s)": 0.0}
    adapt_preds_store[f"{bname} Zero-Shot"] = (b_p, b_t)
    print(f"  {bname} Zero-Shot MAE on B+C = {mae:.4f}")


# ── 2. HMST Partial Fine-Tune (masked loss on B+C) ──
print("\n--- HMST Partial Fine-Tune (masked loss on B+C) ---")
ft_model = _copy.deepcopy(hmst_model_trained)
for p in ft_model.parameters():               p.requires_grad = False
for p in ft_model.blocks[0].parameters():     p.requires_grad = True
for p in ft_model.out_proj.parameters():      p.requires_grad = True

FT_CFG  = {"lr": 5e-4, "max_epochs": 20, "patience": 5, "clip": 1.0}
ft_opt  = torch.optim.Adam([p for p in ft_model.parameters() if p.requires_grad],
                            lr=FT_CFG["lr"])
best_val_ft, best_ft_w, no_imp_ft = float("inf"), None, 0
ft_start = _time_module.time()

for epoch in range(FT_CFG["max_epochs"]):
    ft_model.train()
    for xb, yb, hb, db in ad_train_l:
        xb, yb, hb, db = xb.to(device), yb.to(device), hb.to(device), db.to(device)
        ft_opt.zero_grad()
        pred = ft_model(xb, hb, db)
        loss = nn.functional.mse_loss(pred[:, bc_tensor, :], yb[:, bc_tensor, :])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in ft_model.parameters() if p.requires_grad], FT_CFG["clip"])
        ft_opt.step()

    ft_model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for xb, yb, hb, db in ad_val_l:
            xb, yb, hb, db = xb.to(device), yb.to(device), hb.to(device), db.to(device)
            v_loss += nn.functional.mse_loss(
                ft_model(xb, hb, db)[:, bc_tensor, :], yb[:, bc_tensor, :]).item() * xb.size(0)
    v_loss /= len(ad_val_l.dataset)
    tag = ""
    if v_loss < best_val_ft:
        best_val_ft, best_ft_w, no_imp_ft = v_loss, _copy.deepcopy(ft_model.state_dict()), 0
        tag = " <- best"
    else:
        no_imp_ft += 1
    if (epoch + 1) % 5 == 0 or tag:
        print(f"  FT Epoch {epoch+1:3d} | Val:{v_loss:.5f}{tag}")
    if no_imp_ft >= FT_CFG["patience"]:
        print(f"  [Early Stop]"); break

ft_time = _time_module.time() - ft_start
ft_model.load_state_dict(best_ft_w)
ft_pe, ft_te, ft_p, ft_t = collect_preds(ft_model, ad_test_l, device, bc_mask)
mae, *_ = calculate_metrics(ft_te, ft_pe)
adaptation_results["HMST Fine-Tune"]  = {"MAE": mae, "Time (s)": round(ft_time, 1)}
adapt_preds_store["HMST Fine-Tune"]   = (ft_p, ft_t)
print(f"  HMST Fine-Tune MAE on B+C = {mae:.4f} | Time = {ft_time:.1f}s")


# ── 3. Baselines Full Retrain ──
# All retrain models use the shared TRAIN_CFG (same lr/epochs/patience/clip)
RETRAIN_CFGS = {name: TRAIN_CFG for name in ["All-Grid LSTM", "All-Grid DLinear", "All-Grid PatchTST"]}
RETRAIN_MODELS = {
    "All-Grid LSTM":     AllGridLSTM(hidden_dim=LARGE["hidden_dim"]),
    "All-Grid DLinear":  AllGridDLinear(lookback=LOOKBACK, kernel_size=3),
    "All-Grid PatchTST": AllGridPatchTST(lookback=LOOKBACK, patch_len=4, stride=4,
                                          d_model=LARGE["d_model"],
                                          nhead=LARGE["nhead"],
                                          num_layers=LARGE["num_layers"]),
}
for bname, bmodel in RETRAIN_MODELS.items():
    print(f"\n--- {bname} Full Retrain ---")
    _, bp, bt, bpe, bte, btt, _ = run_training(
        bname, bmodel, RETRAIN_CFGS[bname],
        ad_train_l, ad_val_l, ad_test_l, device, eval_grid_mask=bc_mask)
    mae, *_ = calculate_metrics(bte, bpe)
    key = bname + "\n(Retrain)"
    adaptation_results[key]  = {"MAE": mae, "Time (s)": round(btt, 1)}
    adapt_preds_store[bname] = (bp, bt)
    print(f"  {bname} Retrain MAE on B+C = {mae:.4f} | Time = {btt:.1f}s")


# ── Find best/worst HMST fine-tune grids in B+C ──
ft_bc_mae = np.mean(np.abs(ft_p[:, bc_mask] - ft_t[:, bc_mask]), axis=0)  # per B+C grid
adapt_best_grid  = bc_indices[int(np.argmin(ft_bc_mae))]
adapt_worst_grid = bc_indices[int(np.argmax(ft_bc_mae))]
print(f"\nAdaptation best  grid: {adapt_best_grid} (FT MAE={ft_bc_mae.min():.2f})")
print(f"Adaptation worst grid: {adapt_worst_grid} (FT MAE={ft_bc_mae.max():.2f})")

print("\n--- Adaptation Results Summary ---")
df_adapt = pd.DataFrame(adaptation_results).T
print(df_adapt.to_markdown(floatfmt=".4f"))


In [ ]:
# ── Trajectory: Adaptation Best HMST Grid ──
T_SHOW = 168
gid    = adapt_best_grid

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(adapt_preds_store["HMST Zero-Shot"][1][:T_SHOW, gid],
        color="black", lw=2.5, label="Ground Truth (new pattern)", alpha=0.9)
ax.plot(adapt_preds_store["HMST Zero-Shot"][0][:T_SHOW, gid],
        color="#c0392b", lw=1.8, ls="--", label="HMST Zero-Shot", alpha=0.85)
ax.plot(adapt_preds_store["HMST Fine-Tune"][0][:T_SHOW, gid],
        color="#e74c3c", lw=2.0, ls="-",  label="HMST Fine-Tune", zorder=5)

# Also show Zero-Shot baselines and Retrain baselines
ax.plot(adapt_preds_store["All-Grid LSTM Zero-Shot"][0][:T_SHOW, gid],
        color="#3498db", lw=1.2, ls=":", label="LSTM (Zero-Shot)", alpha=0.7)
ax.plot(adapt_preds_store["All-Grid LSTM"][0][:T_SHOW, gid],
        color="#2980b9", lw=1.4, ls="-.", label="LSTM (Retrain)", alpha=0.8)

ax.plot(adapt_preds_store["All-Grid DLinear Zero-Shot"][0][:T_SHOW, gid],
        color="#2ecc71", lw=1.2, ls=":", label="DLinear (Zero-Shot)", alpha=0.7)
ax.plot(adapt_preds_store["All-Grid DLinear"][0][:T_SHOW, gid],
        color="#27ae60", lw=1.4, ls="-.", label="DLinear (Retrain)", alpha=0.8)

ax.set_title(f"Adaptation — Grid {gid}: HMST Fine-Tune Best Case")
ax.set_xlabel("Hours")
ax.set_ylabel("Population Flow")
ax.legend(loc="upper right", framealpha=0.85)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Trajectory: Adaptation Worst HMST Grid ──
T_SHOW = 168
gid    = adapt_worst_grid

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(adapt_preds_store["HMST Zero-Shot"][1][:T_SHOW, gid],
        color="black", lw=2.5, label="Ground Truth (new pattern)", alpha=0.9)
ax.plot(adapt_preds_store["HMST Zero-Shot"][0][:T_SHOW, gid],
        color="#c0392b", lw=1.8, ls="--", label="HMST Zero-Shot", alpha=0.85)
ax.plot(adapt_preds_store["HMST Fine-Tune"][0][:T_SHOW, gid],
        color="#e74c3c", lw=2.0, ls="-",  label="HMST Fine-Tune", zorder=5)

# Also show Zero-Shot baselines and Retrain baselines
ax.plot(adapt_preds_store["All-Grid LSTM Zero-Shot"][0][:T_SHOW, gid],
        color="#3498db", lw=1.2, ls=":", label="LSTM (Zero-Shot)", alpha=0.7)
ax.plot(adapt_preds_store["All-Grid LSTM"][0][:T_SHOW, gid],
        color="#2980b9", lw=1.4, ls="-.", label="LSTM (Retrain)", alpha=0.8)

ax.plot(adapt_preds_store["All-Grid DLinear Zero-Shot"][0][:T_SHOW, gid],
        color="#2ecc71", lw=1.2, ls=":", label="DLinear (Zero-Shot)", alpha=0.7)
ax.plot(adapt_preds_store["All-Grid DLinear"][0][:T_SHOW, gid],
        color="#27ae60", lw=1.4, ls="-.", label="DLinear (Retrain)", alpha=0.8)

ax.set_title(f"Adaptation — Grid {gid}: HMST Fine-Tune Worst Case")
ax.set_xlabel("Hours")
ax.set_ylabel("Population Flow")
ax.legend(loc="upper right", framealpha=0.85)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Adaptation Speed vs Accuracy Scatter ──
fig, ax = plt.subplots(figsize=(9, 6))
color_map = {
    "HMST Zero-Shot":                  "#e74c3c",
    "Each-Grid LSTM\nZero-Shot":      "#95a5a6",
    "All-Grid LSTM\nZero-Shot":       "#7f8c8d",
    "All-Grid ConvLSTM\nZero-Shot":   "#bdc3c7",
    "All-Grid DLinear\nZero-Shot":    "#34495e",
    "All-Grid PatchTST\nZero-Shot":   "#2c3e50",
    "HMST Fine-Tune":                  "#c0392b",
    "All-Grid LSTM\n(Retrain)":        "#3498db",
    "All-Grid DLinear\n(Retrain)":     "#27ae60",
    "All-Grid PatchTST\n(Retrain)":    "#8e44ad",
}
for key, vals in adaptation_results.items():
    col = color_map.get(key, "gray")
    ax.scatter(vals["Time (s)"], vals["MAE"], color=col,
               s=160, zorder=5, edgecolors="black", lw=0.8)
    ax.annotate(key.replace("\n", " "), (vals["Time (s)"], vals["MAE"]),
                textcoords="offset points", xytext=(6, 4), fontsize=10)

ax.set_xlabel("Adaptation Time (seconds)", fontsize=12)
ax.set_ylabel("MAE on Changed Grids (B+C)", fontsize=12)
ax.set_title("Adaptation Speed vs. Accuracy (lower-left = faster and more accurate)", fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
